In [58]:
import os
import sys

import json
import random
import time
import pandas as pd

from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path

In [2]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

import corpus
import api

from dev import rel, print_epi_summary
from data_config import DATA_CONFIG, FEW_SHOT_PATH
from few_shot import get_few_shot_examples

import pipeline

# Config.

### Development Config.

In [3]:
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at 2026-08-09 15:10


### EPI Config.

In [19]:
# ---------- Manual EPI config. entry ----------
EPI_NUM = "006"
DATASET_SPLIT = "train"

# ---------- Config whether EPI config's API should be called again ----------
CALL_API = False

# Corpus Loading

### Load Corpus

In [20]:
# ---------- Get abstracts path ----------
ABSTRACTS_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["abstracts"]
SAMPLE_SIZE = DATA_CONFIG[DATASET_SPLIT]["sample_size"]

# ---------- Load corpus from path ----------
abstracts_corpus = corpus.load_corpus(ABSTRACTS_PATH, sample_size=SAMPLE_SIZE)
print(f"Abstracts dataset length: {len(abstracts_corpus)}")

Abstracts dataset length: 35


### Get Ground Truths

In [21]:
# ---------- Manually Set Ground Truth Path ----------
GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["ground_truths"]

# ---------- Fetch BioRED Ground Truths ----------
abstracts_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, GT_PATH)

In [22]:
# ---------- Turn ground truth DataFrame into structured dict ----------
ground_truths = corpus.get_gt_dict(abstracts_ground_truths)

In [57]:
import pandas as pd
gt_df = pd.DataFrame(list(ground_truths["relations"]))
print(list(gt_df[2].unique()))

['Association', 'Negative_Correlation', 'Positive_Correlation', 'Bind', 'Comparison']


### Few-Shot Construction

In [23]:
# ---------- If FS block does not exist, create FS block, else pass ----------
if not os.path.exists(FEW_SHOT_PATH):

    few_shot_block = get_few_shot_examples(
        path_to_train_set=DATA_CONFIG["train"]["paths"]["abstracts"],
        path_to_train_gts=DATA_CONFIG["train"]["paths"]["ground_truths"],
        biored_train_samples=abstracts_corpus,
        few_shot_export_path=FEW_SHOT_PATH
    )
    print(f"Generated and exported new few-shot block to '{FEW_SHOT_PATH}'")
else:
    with open(FEW_SHOT_PATH) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{FEW_SHOT_PATH}' at {datetime.now().strftime('%Y-%m-%d %H:%M')}.")

Imported existing few-shot block from '../../data/few_shot/few_shot_block.txt' at 2026-08-09 15:11.


### Import BioRED Extraction Guidelines

In [24]:
# ---------- Import BioRED guidelines text file for prompt refinement ----------
with open("../../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

# ---------- Print preview ----------
print(f"{biored_ext_guidelines[:500]}...")

## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover...


# OpenAI Luna API Call

### EPI Setup

In [25]:
# ---------- Create EPI setup dictionary ----------
# - Keys: "dataset", "id", "eval_version", "notes", "reuse_api_call", "prompt"
epi_setup = pipeline.setup_epi(EPI_NUM, DATASET_SPLIT)

# ---------- Print summary ----------
print_epi_summary(
    epi_num=EPI_NUM,
    prompt_version=epi_setup["prompt"]["version"],
    eval_version=epi_setup["eval_version"],
    notes=epi_setup["notes"],
    reuse_api_call=epi_setup["reuse_api_call"]
)

Cell ran at 2026-08-09 15:11 for epi_006
 - Prompt version: v3
 - Evaluation version: v4
 - Notes: Relationship matching now order-invariant across strict/relaxed/cosine.
 - Reuse API Call: True


### API Call

In [26]:
if CALL_API:
    # ---------- Create client ----------
    client = api.create_client()

    # ---------- If prompt version is different from previous EPI, call API, else pass ----------
    if not epi_setup["reuse_api_call"]:

        print(f"Running API call for {epi_setup["id"]}...")

        # Fetch raw prompt template
        prompt_template = epi_setup["prompt"]["template"]

        # Begin timer
        start_time = time.perf_counter()

        # Initiate outputs list
        outputs = []

        # Begin looping through abstract dataset rows - one call per row
        for index, row in abstracts_corpus.iterrows():
            abstract = row["abstract"]

            # Replace prompt template's placeholders with abstract, few_shot & guidelines
            prompt = (
                prompt_template
                .replace("{abstract}", abstract)
                .replace("{few_shot_block}", few_shot_block)
                .replace("{biored_ext_guidelines}", biored_ext_guidelines)
            )

            # Store row's response
            response = client.responses.create(
                model="gpt-5.6-luna",
                input=prompt
            )

            # Append response to outputs list
            outputs.append({
                "pmid": row["pmid"],
                "output": response.output_text
            })

        # End time, store elapsed time & print result
        elapsed_seconds = time.perf_counter() - start_time
        print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts\n")

        print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f" - Prompt verion: {epi_setup["prompt"]["version"]}")
        print(f" - Evalaution verion: {epi_setup["eval_version"]}")
        print(f" - Run notes: {epi_setup["notes"]}")
        print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
    else:
        prev_epi_id = f"epi_{int(EPI_NUM) - 1:03d}"

        with open(f"{DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]}/{prev_epi_id}.json") as f:
            prev_epi_log = json.load(f)

        outputs = prev_epi_log["outputs"]
        prompt_template = prev_epi_log["prompt"]
        elapsed_seconds = prev_epi_log["time_taken"]

        print(f"Reused API call from {prev_epi_id}.")

    output_info = {
        "epi_id": epi_setup["id"],
        "outputs": outputs,
        "time_taken": elapsed_seconds,
        "raw_prompt": prompt_template,
        "epi_notes": epi_setup["notes"],
        "prompt_version": epi_setup["prompt"]["version"],
        "eval_version": epi_setup["eval_version"],
        "export_path": DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]
    }
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (PIPELINE_RUN = False).")

Pipeline run not executed at 2026-08-09 15:11 (PIPELINE_RUN = False).


In [37]:
if CALL_API:
    # ---------- Parse outputs, evaluate extractions, export EPI results ----------
    epi_log = pipeline.process_epi(output_info, ground_truths, dataset=epi_setup["dataset"])
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (CALL_API = False).")

Pipeline run not executed at 2026-08-09 15:31 (CALL_API = False).


# Error Analysis

### Summary Table

In [109]:
import error_sampling
rel(error_sampling)
from error_sampling import error_summary

Reloaded 'error_sampling' module at 2026-08-09 16:19.


In [110]:
error_summary(epi_log)

`epi_006` Error Summary:


Metric,Precision,Recall,F1
Entity,0.571,0.929,0.708
Relation,0.511,0.615,0.558


In [28]:
ERROR_SAMPLE_SIZE = 20

In [29]:
# ---------- If EPI API call does not exist in kernel state, import existing EPI log
if not CALL_API:
    try:
        with open(f"../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json") as f:
            epi_log = json.load(f)
    except:
        raise ValueError(
            f"EPI log cannot be imported: "
            f"'../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json' does not exist."
        )

### Relations

In [30]:
# ---------- False positives ----------
relations_fp = epi_log["errors"]["errors"]["relations"]["false_positives"]

print(f"Count: {len(relations_fp)}")

if SAMPLE_SIZE < len(relations_fp):
    print(f"Sample size: {ERROR_SAMPLE_SIZE}")
    display(random.sample(relations_fp, ERROR_SAMPLE_SIZE))
else:
    display(relations_fp)

Count: 174
Sample size: 20


[[19918264, 'argarg genotype', 'Association', 'prostate cancer'],
 [28428256, 'pga2', 'Negative_Correlation', 'acute lung injury'],
 [15970799, '*15+c1007g', 'Negative_Correlation', 'estrone-3-sulfate'],
 [16288197, 'congenital microcoria', 'Positive_Correlation', 'glaucoma'],
 [15970799, '521t>c', 'Association', 'slco1b1*5'],
 [19108278, '(+)-propranolol', 'Negative_Correlation', 'isoprenaline'],
 [21163864, 'agt', 'Association', 't704c'],
 [15970799, 'slco1b1*5', 'Association', 'oatp1b1'],
 [20431083, 'warfarin', 'Positive_Correlation', 'ich'],
 [24477591, 'p22phox', 'Bind', 'nadph oxidases'],
 [15970799, '*15+c1007g', 'Negative_Correlation', 'pravastatin'],
 [16120104, '111g allele', 'Association', 'asps'],
 [19108278, '(-)-propranolol', 'Negative_Correlation', 'ouabain'],
 [21163864, 'm235t', 'Positive_Correlation', 'shcm'],
 [18808529, 'isoproterenol', 'Negative_Correlation', 'gamma-sarcoglycan'],
 [28428256, 'prostaglandin a2', 'Bind', 'ep4'],
 [28428256, 'pga2', 'Positive_Correl

In [31]:
# ---------- False negatives ----------
relations_fn = epi_log["errors"]["errors"]["relations"]["false_negatives"]

print(f"Count: {len(relations_fn)}")

if SAMPLE_SIZE < len(relations_fn):
    print(f"Sample size: {ERROR_SAMPLE_SIZE}")
    display(random.sample(relations_fn, ERROR_SAMPLE_SIZE))
else:
    display(relations_fn)

Count: 114
Sample size: 20


[[16120104, '111g', 'Positive_Correlation', 'advanced sleep phase syndrome'],
 [10491763, 'glucose', 'Association', 'type ii diabetes'],
 [25305591,
  'experimental autoimmune encephalomyelitis',
  'Association',
  'vasoactive intestinal peptide'],
 [28512644, 'ccl4', 'Association', 'erysipelas'],
 [16288197, 'congenital microcoria', 'Association', 'optineurin'],
 [20510337, 'coenzyme q10', 'Negative_Correlation', 'superoxide dismutase'],
 [17975693, 'norfloxacin', 'Positive_Correlation', 'anxiogenic'],
 [18768591,
  'serum- and glucocorticoid-inducible kinase 1',
  'Association',
  'doxorubicin'],
 [18827003, 'hypersensitivity', 'Association', 'glucocorticoid receptor'],
 [16288197, 'congenital microcoria', 'Association', 'mcor'],
 [25305591,
  'pituitary adenylyl cyclase-activating polypeptide',
  'Association',
  'experimental autoimmune encephalomyelitis'],
 [18827003,
  'aspartic acid to histidine substitution at amino acid position 401',
  'Positive_Correlation',
  'metabolic syn

### Entities

In [32]:
# ---------- False positives ----------
entities_fp = epi_log["errors"]["errors"]["entities"]["false_positives"]

print(f"Count: {len(entities_fp)}")

if SAMPLE_SIZE < len(entities_fp):
    print(f"Sample size: {ERROR_SAMPLE_SIZE}")
    display(random.sample(entities_fp, ERROR_SAMPLE_SIZE))
else:
    display(entities_fp)

Count: 178
Sample size: 20


[[28428256, 'nfkb', 'GeneOrGeneProduct'],
 [28512644, 'bullous erysipelas', 'DiseaseOrPhenotypicFeature'],
 [16574712, 'ecstasy', 'ChemicalEntity'],
 [19918264, 'arggly genotype', 'SequenceVariant'],
 [28428256, 'pga2', 'ChemicalEntity'],
 [19521089, 'pd', 'DiseaseOrPhenotypicFeature'],
 [28512644, 'erysipelas', 'DiseaseOrPhenotypicFeature'],
 [24914936, 'tsh', 'ChemicalEntity'],
 [21163864, 'hcm', 'DiseaseOrPhenotypicFeature'],
 [28512644, 'il-2ralpha', 'GeneOrGeneProduct'],
 [21771880, 'fat mass', 'DiseaseOrPhenotypicFeature'],
 [24743235, 'il-6', 'ChemicalEntity'],
 [19918264, 'argarg genotype', 'SequenceVariant'],
 [17192049, 'w2/m2', 'SequenceVariant'],
 [21163864, 't allele', 'SequenceVariant'],
 [10491763,
  'type ii (non-insulin-dependent) diabetes mellitus',
  'DiseaseOrPhenotypicFeature'],
 [16200390, '5-ht', 'ChemicalEntity'],
 [16288197, 'optn', 'GeneOrGeneProduct'],
 [28428256, 've-cadherin', 'GeneOrGeneProduct'],
 [16737910, 'translocation-ets-leukemia', 'GeneOrGeneProduc

In [33]:
# ---------- False negatives ----------
entities_fn = epi_log["errors"]["errors"]["entities"]["false_negatives"]

print(f"Count: {len(entities_fn)}")

if SAMPLE_SIZE < len(entities_fn):
    print(f"Sample size: {ERROR_SAMPLE_SIZE}")
    display(random.sample(entities_fn, ERROR_SAMPLE_SIZE))
else:
    display(entities_fn)

Count: 18


[[16737910, 'cancer', 'DiseaseOrPhenotypicFeature'],
 [15686794, 'antiarrhythmic drug', 'ChemicalEntity'],
 [18827003, 'glucocorticoid', 'ChemicalEntity'],
 [18768591, 'weight gain', 'DiseaseOrPhenotypicFeature'],
 [24743235, 'il-10', 'GeneOrGeneProduct'],
 [15970799, 'tritium', 'ChemicalEntity'],
 [24743235, 'il-3', 'GeneOrGeneProduct'],
 [28411266, 'sulfonylurea receptor', 'GeneOrGeneProduct'],
 [16288197, 'pigmentation', 'DiseaseOrPhenotypicFeature'],
 [18768591, 'salt', 'ChemicalEntity'],
 [24743235, 'csf-1', 'GeneOrGeneProduct'],
 [28411266, 'insulin', 'GeneOrGeneProduct'],
 [20510337, 'lipid', 'ChemicalEntity'],
 [20683499, 'neurodegenerative diseases', 'DiseaseOrPhenotypicFeature'],
 [18808529, 'oxygen', 'ChemicalEntity'],
 [24743235, 'cd11c', 'GeneOrGeneProduct'],
 [18827003, 'cortisol', 'ChemicalEntity'],
 [28348168, 'degenerative disease', 'DiseaseOrPhenotypicFeature']]